# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abuhussein1504/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

---

### Finding 1 — "What Predicts Health?" (p. 27, ML Appendix)

The paper trains a Random Forest to predict **Health Score** from ten inputs and reports feature
importance: Average Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8% — everything else
(Clicks, Sessions, Content Age, Word Count, Days Visible, AI Sessions) rounds to ~0-2%. The paper
itself is careful here — it says the ranking is "descriptive rather than causal" and that the
target is "partly constructed from some of these inputs."

**My methodology question — where does the label come from?**
Health Score is *defined*, by formula, as `Impressions (30) + Position (30) + CTR (20) + Scroll
Depth (20)`. Those four inputs are exactly the four features that dominate the importance chart,
and they sum to ~98% importance (43+32+15+8, verified below). That isn't the model discovering
what predicts health — it's the model reconstructing the scoring formula from its own ingredients,
the way a Random Forest would perfectly "predict" a sum if given the addends. The paper's word
"partly" is doing a lot of work here: given that the four scoring inputs alone explain ~98% of the
importance, "entirely" (by construction) would be the more accurate word for this specific chart.
Asked respectfully, as I'd want it asked of mine: would the appendix be stronger if it separated
the two things it's currently blending — (a) "here is how our own formula weights its parts"
(trivially true, no model needed) from (b) "here is what predicts a target we did NOT build from
these same parts" (e.g., future clicks or `trend_direction`, which the paper already treats
correctly as an *observed* target elsewhere)? Finding (b) would carry a claim the chart on p. 27
doesn't currently support.

### Finding 2 — "What Predicts Growth?" (p. 29 + Methodology, p. 36)

The paper reports a Logistic Regression "71% holdout accuracy" separating growing from declining
pages, on an 80/20 split, over a 61.8K-row ML feature-vector snapshot (p. 36, Methodology).

**My methodology question — does the validation design carry the claim?**
The Methodology section is honest about what it *doesn't* report: no p-values, no confidence
intervals (stated directly on p. 36). But it also doesn't say whether the 80/20 split was a random
row split or a split grouped by client/brand, and it doesn't report the base rate of
growing-vs-declining pages in that 61.8K sample — both needed to know what "71%" is actually worth.
This isn't a hypothetical concern: Section 2 below reruns *my own* Week-5 model both ways on
comparable data, and the naive random split reports Precision@50 = 0.860 while the honest,
client-grouped split reports 0.720 on the *same* model and *same* metric — an 0.14 gap purely from
how rows were assigned to train/test, because pages from the same client share hidden character a
random split lets the model partly memorize. I can't tell from the paper which kind of split
produced 71%, so I can't tell how much of that number is signal versus client memorization. A
methodology note naming the split unit (row-random vs. grouped-by-brand) and the base rate would
let a reader answer that themselves — the same fix I'm applying to my own claim in Section 4.


In [5]:
import subprocess, sys
import os
from pathlib import Path

# --- Start of added setup for PDF --- #
REPO_URL = "https://github.com/abuhussein1504/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
PDF_PATH_IN_REPO = "docs/flyrank-seo-research-march-2026.pdf"

# Clone the repository if it doesn't exist
if not Path(REPO_DIR).is_dir():
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print("Repository cloned.")

# Construct the full path to the PDF file
PDF = Path(REPO_DIR) / PDF_PATH_IN_REPO
if not PDF.exists():
    raise FileNotFoundError(f"PDF file not found after cloning: {PDF}")

# --- End of added setup for PDF --- #

try:
    import pdfplumber
except ImportError:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pdfplumber"],
                       check=True, capture_output=True)
    except subprocess.CalledProcessError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", "pdfplumber"],
                       check=True, capture_output=True)
    import pdfplumber

with pdfplumber.open(PDF) as pdf:
    p27 = pdf.pages[26].extract_text()
    p29 = pdf.pages[28].extract_text()
    p36 = pdf.pages[35].extract_text()

print("--- p.27 excerpt (Health Score feature importance) ---")
for line in p27.splitlines():
    if any(k in line for k in ["Average Position", "Impressions", "Scroll Depth", "CTR", "partly constructed"]):
        print(line)

# The paper's own numbers: do the four Health-Score-defining inputs really dominate importance?
health_score_inputs_importance = {"Average Position": 43, "Impressions": 32, "Scroll Depth": 15, "CTR": 8}
total = sum(health_score_inputs_importance.values())
print(f"\nSum of importance for the 4 features that DEFINE Health Score: {total}% of the chart")

print("\n--- p.29 excerpt (Growth model headline) ---")
for line in p29.splitlines():
    if "71%" in line or "holdout accuracy" in line:
        print(line)

print("\n--- p.36 excerpt (Methodology — what's disclosed / not disclosed) ---")
for line in p36.splitlines():
    if any(k in line for k in ["split", "p-values", "confidence intervals", "80/20", "61.8"]):
        print(line)


--- p.27 excerpt (Health Score feature importance) ---
itself is partly constructed from some of these inputs, so importance is descriptive rather than causal.
Average Position 43
Impressions 32
Scroll Depth 15
CTR 8
Average Position is the #1 predictor of health score at 43% importance, followed by Impressions (32%) and
Scroll Depth (15%). This ranking shows which features the model uses most, though note that health score is partly

Sum of importance for the 4 features that DEFINE Health Score: 98% of the chart

--- p.29 excerpt (Growth model headline) ---
Logistic regression (71% holdout accuracy) describing which sampled features separate growing from

--- p.36 excerpt (Methodology — what's disclosed / not disclosed) ---
61.8K content pieces with sessions > 0 and impressions > 0 in the local feature-vector snapshot. Sklearn: K-Means (k=5), Random Forest
(80/20 split), Logistic Regression (80/20 split), PCA (2 components), and a shallow Decision Tree (80/20 split). ML pages remain e

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

---

**Setup.** Same starter CSV, same eligible population, same target (`is_declining` =
`trend_direction == "down"`, observed not defined), same final feature set as `w05_model.ipynb`
(post-leakage-check — `impressions_last_30d`/`impressions_prev_30d` already excluded there; see
Section 3 for why). `content_id` and `client_id` stay out of the feature matrix — context only.

**Before:** a plain 5-fold `KFold` over rows, shuffled, no grouping. Rows from the same client can
land in both train and test — the model can partly memorize a client's baseline behavior instead
of learning a transferable decline pattern.

**After:** `GroupKFold` by `client_id`, 5 folds — the same honest split `w05_model.ipynb` already
used. Every fold tests on clients the model never trained on.

Both use Logistic Regression (the stronger of the two Week-5 methods on this feature set), same
metric (Precision@50), same K, same data — only the split unit changes.


In [6]:
import os, sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd

REPO_URL = "https://github.com/abuhussein1504/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                   check=True, capture_output=True)
except subprocess.CalledProcessError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", "-r", "requirements.txt"],
                   check=True, capture_output=True)
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# Same eligible population as w04/w05: only pages we actually have signal on.
eligible_mask = (df["impression_tier"] != "no_data") & (df["avg_position"] > 0)
elig = df.loc[eligible_mask].reset_index(drop=True).copy()

# Same missingness-flags-before-fillna pattern as w05 (data dictionary: missingness follows content_type).
elig["has_keyword_data"] = elig["search_volume"].notna().astype(int)
elig["has_word_count"] = elig["word_count"].notna().astype(int)

# Same final, post-leakage-check feature list as w05_model.ipynb.
numeric_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "clicks_last_30d", "sessions_last_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
flag_cols = ["has_keyword_data", "has_word_count"]
cat_cols = ["content_type", "main_intent"]
elig[cat_cols] = elig[cat_cols].fillna("unknown")
cat_dummies = pd.get_dummies(elig[cat_cols], prefix=cat_cols)

X = pd.concat([elig[numeric_cols].fillna(0), elig[flag_cols], cat_dummies], axis=1)
y = elig["is_declining"].values
groups = elig["client_id"].values

print(f"rows: {len(X):,}  |  clients: {len(np.unique(groups))}  |  base rate: {y.mean():.3f}")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, GroupKFold

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K = 50
base_rate = y.mean()

def run_cv(splitter, split_args):
    oof = np.zeros(len(X))
    for tr, te in splitter.split(*split_args):
        scaler = StandardScaler()
        Xtr_s, Xte_s = scaler.fit_transform(X.iloc[tr]), scaler.transform(X.iloc[te])
        m = LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)
        m.fit(Xtr_s, y[tr])
        oof[te] = m.predict_proba(Xte_s)[:, 1]
    return oof

# BEFORE: naive random row split — no grouping.
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_naive = run_cv(kf, (X,))
p_naive = precision_at_k(oof_naive, y, K)

# Show WHY the naive split is dishonest: how many test-fold clients also appear in that fold's train set?
kf_check = KFold(n_splits=5, shuffle=True, random_state=42)
overlaps = []
for tr, te in kf_check.split(X):
    overlaps.append(len(set(groups[tr]) & set(groups[te])))
print(f"naive split: avg clients appearing in BOTH train and test per fold: {np.mean(overlaps):.1f} of {len(np.unique(groups))}")

# AFTER: GroupKFold by client_id — the honest split, same one w05_model.ipynb already used.
gkf = GroupKFold(n_splits=5)
oof_grouped = run_cv(gkf, (X, y, groups))
p_grouped = precision_at_k(oof_grouped, y, K)

before_after = pd.DataFrame({
    "split": ["Base rate (random guessing)", "BEFORE — naive random row KFold", "AFTER — GroupKFold by client_id"],
    f"precision@{K}": [round(float(base_rate), 3), round(float(p_naive), 3), round(float(p_grouped), 3)],
})
print()
print(before_after.to_string(index=False))
print(f"\nGap from split choice alone: {p_naive - p_grouped:+.3f} precision@{K} points")


rows: 28,795  |  clients: 31  |  base rate: 0.564
naive split: avg clients appearing in BOTH train and test per fold: 30.6 of 31

                          split  precision@50
    Base rate (random guessing)         0.564
BEFORE — naive random row KFold         0.860
AFTER — GroupKFold by client_id         0.720

Gap from split choice alone: +0.140 precision@50 points


**Reading the before/after.** The naive random split reports Precision@50 = 0.860; the honest,
client-grouped split reports 0.720 — a 0.14-point drop from changing nothing but which rows are
allowed to sit in both train and test. In the naive split every one of the 31 clients shows up in
both the train and test side of each fold (printed above), so part of the "0.860" was the model
recognizing a client it had already seen, not a transferable decline pattern. The grouped number
(0.720) matches `w05_model.ipynb`'s reported result exactly, because that notebook already used
`GroupKFold` — this section exists to show *why* that choice mattered, with the dishonest
alternative sitting right next to it instead of just asserted. The honest number is still clearly
above both the base rate (0.564) and the Week-4 rule (0.44, from `w05_model_metrics.json`) — the
model is doing real work, just less of it than the naive split implied.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

---

Running the `hunting-leakage-and-validating` checklist against the exact feature set used in
Section 2 (identical to `w05_model.ipynb`'s final list):

1. **Label-derived features.** `w05_model.ipynb` already caught and removed `impressions_last_30d`
   / `impressions_prev_30d` — those two columns are the literal source columns of `trend_pct`
   (and therefore of `trend_direction` and my label): `corr((last-prev)/prev, trend_pct) =
   0.99999998`. They are confirmed absent from `X` below. To make sure the *test itself* still
   works this week (not just trust last week's result), I deliberately inject a known-leaky column
   — `trend_pct` itself, the direct source of the label — and confirm the score shoots toward 1.0.
   If it hadn't, the checking tool would be broken, not the model.
2. **Future/overlapping windows.** This is a single trailing-90-day snapshot, not a daily panel, so
   there's no `report_date` to draw a timeline against directly — but the same principle applies at
   column level: every remaining feature (90-day aggregates, `days_since_last_update`,
   `content_age_days`, `clicks_last_30d`/`clicks_prev_30d`, `sessions_last_30d`/`sessions_prev_30d`)
   describes the page's state *up to* the export, never a value computed from the outcome window
   used to build `trend_direction`. The two columns that *were* in that outcome window
   (`impressions_last_30d`/`prev_30d`) are the ones already removed in point 1.
3. **Decision-derived features (product flags).** The starter CSV has no health-score or
   optimization-flag column at all — those only exist in the research paper's dataset, not here —
   so there's nothing of that kind to accidentally use as a feature in this project.
4. **Context IDs.** `content_id` / `client_id` confirmed absent from the feature matrix — grouping
   only, per the data contract.
5. **Missingness pattern.** Re-checked that missingness still follows `content_type` (not random) —
   this is why the `has_keyword_data`/`has_word_count` flags exist before any `fillna(0)`.


In [7]:
# 1. Deliberate leak injection — the checking tool itself must be sensitive to a known leak.
X_poison = X.copy()
X_poison["trend_pct_LEAK_TEST"] = elig["trend_pct"].fillna(0).values

oof_poison = np.zeros(len(X))
for tr, te in GroupKFold(n_splits=5).split(X_poison, y, groups):
    scaler = StandardScaler()
    Xtr_s, Xte_s = scaler.fit_transform(X_poison.iloc[tr]), scaler.transform(X_poison.iloc[te])
    m = LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)
    m.fit(Xtr_s, y[tr])
    oof_poison[te] = m.predict_proba(Xte_s)[:, 1]
p_poison = precision_at_k(oof_poison, y, K)

print("--- Leak-detector sanity check ---")
print(f"precision@{K} honest features only         : {p_grouped:.3f}")
print(f"precision@{K} WITH trend_pct injected on top: {p_poison:.3f}  <- should jump toward 1.0")
assert p_poison > p_grouped + 0.15, "checking tool looks broken: injecting the label's own source didn't move the score"
print("Checking tool is working: a known leak is detected.\n")

# 2/4. Confirm the already-removed / never-included columns really are absent from X.
suspect_cols = ["impressions_last_30d", "impressions_prev_30d", "content_id", "client_id",
                 "trend_direction", "trend_pct"]
present = [c for c in suspect_cols if c in X.columns]
print(f"Suspect columns still present in final feature matrix: {present if present else 'NONE'}")

# 5. Missingness pattern by content_type — confirms a blind fillna(0) would have been unsafe.
print("\nMissing rate by content_type (word_count, search_volume):")
print(elig.groupby("content_type")[["word_count", "search_volume"]].apply(lambda d: d.isna().mean()).round(3))


--- Leak-detector sanity check ---
precision@50 honest features only         : 0.720
precision@50 WITH trend_pct injected on top: 1.000  <- should jump toward 1.0
Checking tool is working: a known leak is detected.

Suspect columns still present in final feature matrix: NONE

Missing rate by content_type (word_count, search_volume):
                    word_count  search_volume
content_type                                 
comparison article       0.000          0.000
feedly article           0.000          1.000
keyword article          0.288          0.013


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

---

**Original (Week-5) sentence, before this audit:**
> "Our model is right 72% of the time, which is better than the old rule, so it will help any
> customer next month, and fixing the pages it points to will stop them from declining."

**What's wrong with it, one problem at a time:**
- "72% of the time" needs the base rate next to it — random guessing already gets 56.4% (the
  eligible slice is more decline-heavy than not), so the honest gap is closer to +16 points, not
  "72%" standing alone.
- "will help ANY customer" is too strong — the grouped split tests transfer to held-out clients on
  average, not every client individually; I haven't checked per-client variance.
- "next month" was never tested going forward in time on this dataset — the starter CSV is one
  snapshot, not a panel, so there's no time-based holdout here, only the client-grouped one.
- "will stop them from declining" claims causation. Nothing here is an experiment — no pages were
  actually refreshed and re-measured. This is a ranking model on cross-sectional data.

**Rewritten, using the four safe words:**
> We **observed** that, on this snapshot's eligible pages, a Logistic Regression ranked pages by
> decline risk **measurably** better than both random ordering and the Week-4 rule: Precision@50
> of 0.720 under a client-grouped holdout, against a base rate of 0.564 and a Week-4-rule score of
> 0.440 (`w05_model_metrics.json`). The **directional** pattern — recent clicks/sessions dropping
> against a healthier longer-run baseline — held up when tested on clients the model never trained
> on, though the same random-split test on this data showed how much that number can inflate
> (0.860) when client identity leaks across train/test. This is **decision-support**, not a
> guarantee: it helps a reviewer pick which pages to look at first out of a large queue. It does
> not establish that reviewing or refreshing those pages will fix the decline — that would need an
> actual before/after experiment, which this notebook does not run. It also has not yet been tested
> against a future month of data, only against held-out clients from the same snapshot.


In [8]:
# Receipt for the rewritten claim — the exact numbers it cites, computed in this notebook run.
claim_receipt = {
    "k": K,
    "base_rate": round(float(base_rate), 3),
    "precision_at_k": {
        "week4_baseline_rule": 0.440,   # from work/outputs/w05_model_metrics.json, same eligible population
        "logistic_regression_grouped_honest": round(float(p_grouped), 3),
        "logistic_regression_naive_random_split": round(float(p_naive), 3),
    },
    "leak_detector_sanity_check": {
        "precision_at_k_with_injected_trend_pct": round(float(p_poison), 3),
        "note": "confirms the leakage test is sensitive; trend_pct is NOT a real feature, it's the label source",
    },
    "claim_language": ["observed", "measured", "directional", "decision-support"],
    "not_claimed": ["causation", "guaranteed per-client benefit", "forward-in-time performance"],
}

import json
from pathlib import Path
out_path = Path("work/outputs/w06_validation_audit.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    json.dump(claim_receipt, f, indent=2)

print(json.dumps(claim_receipt, indent=2))
print(f"\nWrote {out_path}")


{
  "k": 50,
  "base_rate": 0.564,
  "precision_at_k": {
    "week4_baseline_rule": 0.44,
    "logistic_regression_grouped_honest": 0.72,
    "logistic_regression_naive_random_split": 0.86
  },
  "leak_detector_sanity_check": {
    "precision_at_k_with_injected_trend_pct": 1.0,
    "note": "confirms the leakage test is sensitive; trend_pct is NOT a real feature, it's the label source"
  },
  "claim_language": [
    "observed",
    "measured",
    "directional",
    "decision-support"
  ],
  "not_claimed": [
    "causation",
    "guaranteed per-client benefit",
    "forward-in-time performance"
  ]
}

Wrote work/outputs/w06_validation_audit.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
